# BF16, gradient accumulation, and GPU training

This notebook runs two microbatches through one efficient DeepSeek+MTP optimizer step and compares attention checkpointing behavior.

In [ ]:
import sys
from pathlib import Path

project_root = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / 'src' / 'model').is_dir()),
    None,
)
if project_root is None:
    raise RuntimeError('Could not find the repository root')
sys.path.insert(0, str(project_root))

import torch

from src.model import DeepSeekConfig, DeepSeekWithMTP
from src.training import (
    MoELoadBalancer,
    create_optimizer,
    efficient_train_step,
    profile_train_step,
)

torch.manual_seed(26)

In [ ]:
config = DeepSeekConfig(
    vocab_size=256, max_seq_len=32, dim=64,
    n_layers=3, n_dense_layers=1, n_heads=4,
    q_lora_rank=16, kv_lora_rank=8,
    qk_nope_head_dim=8, qk_rope_head_dim=4, v_head_dim=8,
    inter_dim=128, moe_inter_dim=32,
    n_routed_experts=4, n_shared_experts=1,
    n_activated_experts=2,
)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DeepSeekWithMTP(config, mtp_weight=0.3).to(device)
model.set_activation_checkpointing(True)
optimizer = create_optimizer(model)
microbatches = [
    torch.randint(0, config.vocab_size, (2, 17), device=device)
    for _ in range(2)
]
print('device:', device)

In [ ]:
with MoELoadBalancer(model, gamma=0.01) as balancer:
    metrics = profile_train_step(
        lambda: efficient_train_step(
            model, optimizer, balancer, microbatches,
            step=1, total_steps=5, warmup_steps=1,
        ),
        tokens_per_step=2 * 2 * 16,
    )

for key, value in metrics.items():
    print(f'{key}: {value}')

The two microbatches share one optimizer update. Routing assignments are collected across both forwards and balancing biases are updated once. On a BF16-capable CUDA device, the efficient step automatically uses BF16 autocast.

In [ ]:
assert torch.isfinite(torch.tensor(metrics['loss']))
assert metrics['step_seconds'] > 0
assert metrics['tokens_per_second'] > 0
assert len(metrics['load_ratios']) == 3
print('efficient training checks passed')